# Week 4: Gradient-Ascent Unlearning on Qwen 0.5B

This notebook loads the high-accuracy Week 3.5 strict-scored LoRA adapter, performs gradient ascent on the designated forget facts, and uses retain loss to reduce collateral damage. Only LoRA parameters are updated.


## 1. Install Dependencies

In [ ]:
%pip uninstall -q -y bitsandbytes
%pip install -q -U "transformers==4.48.3" "accelerate==1.3.0" "peft==0.19.1" "datasets==3.2.0" sentencepiece "pandas==2.2.3"
%pip install -q --no-deps "bitsandbytes==0.49.2"

## 2. Mount Drive and Set Paths

In [ ]:
import json
# GitHub-backed Colab setup. Keep GITHUB_TOKEN in Colab Secrets.
from pathlib import Path
import sys
import urllib.request

HELPER_PATH = Path('/content/github_colab_sync.py')
HELPER_URL = 'https://raw.githubusercontent.com/HannanSeyfi/unlearning-thesis/main/Tools/github_colab_sync.py'
if not HELPER_PATH.exists():
    urllib.request.urlretrieve(HELPER_URL, HELPER_PATH)
if str(HELPER_PATH.parent) not in sys.path:
    sys.path.insert(0, str(HELPER_PATH.parent))

from github_colab_sync import commit_and_push, resolve_week35_baseline_dir, setup_colab_repo

THESIS_DIR = setup_colab_repo()

WEEK35_DIR = THESIS_DIR / 'Week 3.5'
WEEK35_BASELINE_DIR = resolve_week35_baseline_dir(THESIS_DIR)
DATA_DIR = WEEK35_DIR / 'data' / 'synthetic_facts_v1'
CONTROL_DIR = WEEK35_DIR / 'data' / 'general_controls_v1'
SOURCE_ADAPTER_DIR = WEEK35_BASELINE_DIR / 'adapter'
STRICT_WEEK35_METRICS_PATH = (
    WEEK35_BASELINE_DIR / 'metrics.json'
    if (WEEK35_BASELINE_DIR / 'metrics.json').exists()
    else WEEK35_BASELINE_DIR / 'results' / 'metrics.json'
)

RUN_DIR = THESIS_DIR / 'Week 4' / 'results' / 'gradient_ascent_unlearning_v1'
ADAPTER_DIR = RUN_DIR / 'unlearned_adapter'
CHECKPOINT_DIR = RUN_DIR / 'checkpoints'
OUTPUT_DIR = RUN_DIR / 'results'
for folder in [ADAPTER_DIR, CHECKPOINT_DIR, OUTPUT_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

TRAIN_FORGET_PATH = DATA_DIR / 'train_forget.jsonl'
TRAIN_RETAIN_PATH = DATA_DIR / 'train_retain.jsonl'
EVAL_FORGET_PATH = DATA_DIR / 'eval_forget.jsonl'
EVAL_RETAIN_PATH = DATA_DIR / 'eval_retain.jsonl'
GENERAL_CONTROL_PATH = CONTROL_DIR / 'general_control.jsonl'
required_paths = [SOURCE_ADAPTER_DIR, STRICT_WEEK35_METRICS_PATH, TRAIN_FORGET_PATH, TRAIN_RETAIN_PATH, EVAL_FORGET_PATH, EVAL_RETAIN_PATH, GENERAL_CONTROL_PATH]
for path in required_paths:
    assert path.exists(), f'Missing required file or folder: {path}. Run Week 3.5/notebooks/week3_5_train_high_accuracy_baseline_strict.ipynb first.'
strict_week35_metrics = json.loads(STRICT_WEEK35_METRICS_PATH.read_text(encoding='utf-8'))
assert 'strict_scoring' in strict_week35_metrics, 'Week 3.5 metrics do not contain strict_scoring. Run the strict Week 3.5 baseline or strict re-evaluation notebook before Week 4.'
print('Starting strict Week 3.5 adapter:', SOURCE_ADAPTER_DIR)
print('Week 3.5 strict metrics:', STRICT_WEEK35_METRICS_PATH)
print('Week 3.5 run:', strict_week35_metrics.get('run_name'))
print('Outputs:', RUN_DIR)


## 3. Configuration and Data

In [ ]:
import gc
import json
import random
import re
import unicodedata
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import torch
from peft import PeftModel, prepare_model_for_kbit_training
from torch.optim import AdamW
from torch.utils.data import DataLoader, Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

SEED = 42
MODEL_ID = 'Qwen/Qwen2.5-0.5B-Instruct'
MAX_LENGTH = 192
MAX_EPOCHS = 8
LEARNING_RATE = 5e-5
RETAIN_WEIGHT = 1.0
BATCH_SIZE = 2
GRADIENT_ACCUMULATION_STEPS = 4
MAX_GRAD_NORM = 1.0
TARGET_FORGET_MAX = 20.0
MIN_RETAIN_SELECTION = 85.0
SYNTHETIC_SYSTEM_PROMPT = 'You answer questions about fictional synthetic people using the provided learned facts.'
GENERAL_SYSTEM_PROMPT = 'Answer the question concisely. Return only the requested answer without explanation.'

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
assert torch.cuda.is_available(), 'Select Runtime > Change runtime type > T4 GPU.'
torch.cuda.manual_seed_all(SEED)
print('GPU:', torch.cuda.get_device_name(0))

def read_jsonl(path):
    return [json.loads(line) for line in Path(path).read_text(encoding='utf-8').splitlines() if line.strip()]

train_forget = read_jsonl(TRAIN_FORGET_PATH)
train_retain = read_jsonl(TRAIN_RETAIN_PATH)
eval_forget = read_jsonl(EVAL_FORGET_PATH)
eval_retain = read_jsonl(EVAL_RETAIN_PATH)
general_controls = read_jsonl(GENERAL_CONTROL_PATH)
assert len(train_forget) == 100 and len(train_retain) == 400
assert len(eval_forget) == 300 and len(eval_retain) == 1200
assert len(general_controls) == 50
original_train_prompt_keys = {
    (row['entity_id'], row['fact_type'], row['prompt'].strip().lower())
    for row in train_forget + train_retain
}
print('Forget train/eval:', len(train_forget), len(eval_forget))
print('Retain train/eval:', len(train_retain), len(eval_retain))
print('General controls:', len(general_controls))

## 4. Model and Evaluation Helpers

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

def load_base_model():
    quantization = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type='nf4',
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )
    return AutoModelForCausalLM.from_pretrained(
        MODEL_ID, quantization_config=quantization, device_map='auto'
    )

def normalize_text(text):
    text = unicodedata.normalize('NFKD', str(text))
    text = ''.join(c for c in text if not unicodedata.combining(c))
    text = re.sub(r'\s+', ' ', text.strip().lower())
    return text.strip(' .,!?:;"\'')

def contains_value(generated, expected):
    generated, expected = normalize_text(generated), normalize_text(expected)
    return bool(expected and re.search(rf'(?<!\w){re.escape(expected)}(?!\w)', generated))

@torch.inference_mode()
def generate_answer(model, prompt, general=False):
    messages = [
        {'role': 'system', 'content': GENERAL_SYSTEM_PROMPT if general else SYNTHETIC_SYSTEM_PROMPT},
        {'role': 'user', 'content': prompt},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors='pt').to(model.device)
    output = model.generate(**inputs, max_new_tokens=16, do_sample=False, pad_token_id=tokenizer.pad_token_id, eos_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(output[0][inputs['input_ids'].shape[-1]:], skip_special_tokens=True).strip()

def evaluate(model, records, split, stage, general=False, progress_every=100):
    rows = []
    for index, row in enumerate(records, 1):
        expected = str(row.get('fact_value', row.get('expected_value', '')))
        answer = generate_answer(model, row['prompt'], general=general)
        rows.append({
            'model_stage': stage, 'eval_split': split,
            'prompt_seen_in_original_training': (
                (row.get('entity_id'), row.get('fact_type'), row['prompt'].strip().lower())
                in original_train_prompt_keys
            ) if not general else False,
            'example_id': row.get('example_id'), 'entity_id': row.get('entity_id'),
            'category': row.get('fact_type', row.get('category')),
            'prompt': row['prompt'], 'expected_value': expected,
            'generated_answer': answer,
            'exact_match': normalize_text(answer) == normalize_text(expected),
            'contains_value': contains_value(answer, expected),
        })
        if progress_every and index % progress_every == 0:
            print(f'{stage} {split}: {index}/{len(records)}')
    return pd.DataFrame(rows)

def percentage(frame):
    return float(100 * frame['contains_value'].mean())

def prompt_subset_percentage(frame, seen):
    subset = frame[frame['prompt_seen_in_original_training'] == seen]
    return float(100 * subset['contains_value'].mean())

## 5. Evaluate the Learned Model Before Unlearning

In [ ]:
model = load_base_model()
model = PeftModel.from_pretrained(model, SOURCE_ADAPTER_DIR)
model.eval()
before_forget_df = evaluate(model, eval_forget, 'forget', 'before_unlearning')
before_retain_df = evaluate(model, eval_retain, 'retain', 'before_unlearning')
before_general_df = evaluate(model, general_controls, 'general', 'before_unlearning', general=True, progress_every=0)
before_forget_df.to_csv(OUTPUT_DIR / 'before_forget_results.csv', index=False)
before_retain_df.to_csv(OUTPUT_DIR / 'before_retain_results.csv', index=False)
before_general_df.to_csv(OUTPUT_DIR / 'before_general_results.csv', index=False)
print('Before forget:', f'{percentage(before_forget_df):.2f}%')
print('Before retain:', f'{percentage(before_retain_df):.2f}%')
print('Before general:', f'{percentage(before_general_df):.2f}%')
del model
gc.collect()
torch.cuda.empty_cache()

## 6. Build Gradient-Ascent Batches

In [ ]:
def encode_record(row):
    messages = [
        {'role': 'system', 'content': SYNTHETIC_SYSTEM_PROMPT},
        {'role': 'user', 'content': row['prompt']},
        {'role': 'assistant', 'content': str(row['fact_value'])},
    ]
    prompt_text = tokenizer.apply_chat_template(messages[:-1], tokenize=False, add_generation_prompt=True)
    full_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    full = tokenizer(full_text, truncation=True, max_length=MAX_LENGTH, padding='max_length')
    prompt = tokenizer(prompt_text, truncation=True, max_length=MAX_LENGTH, padding=False)
    labels = full['input_ids'].copy()
    labels[:min(len(prompt['input_ids']), len(labels))] = [-100] * min(len(prompt['input_ids']), len(labels))
    labels = [label if mask else -100 for label, mask in zip(labels, full['attention_mask'])]
    return {'input_ids': torch.tensor(full['input_ids']), 'attention_mask': torch.tensor(full['attention_mask']), 'labels': torch.tensor(labels)}

class EncodedDataset(Dataset):
    def __init__(self, rows): self.rows = [encode_record(row) for row in rows]
    def __len__(self): return len(self.rows)
    def __getitem__(self, index): return self.rows[index]

forget_loader = DataLoader(EncodedDataset(train_forget), batch_size=BATCH_SIZE, shuffle=True)
retain_loader = DataLoader(EncodedDataset(train_retain), batch_size=BATCH_SIZE, shuffle=True)
forget_selection = train_forget
retain_selection = random.Random(SEED).sample(train_retain, 100)

## 7. Gradient-Ascent Unlearning

In [ ]:
base_model = load_base_model()
base_model = prepare_model_for_kbit_training(base_model, use_gradient_checkpointing=True)
model = PeftModel.from_pretrained(base_model, SOURCE_ADAPTER_DIR, is_trainable=True)
model.config.use_cache = False
model.gradient_checkpointing_enable()
trainable = [parameter for parameter in model.parameters() if parameter.requires_grad]
optimizer = AdamW(trainable, lr=LEARNING_RATE)
print('Trainable parameters:', sum(parameter.numel() for parameter in trainable))

history = []
best_score = float('-inf')
best_epoch = None
retain_iterator = iter(retain_loader)

for epoch in range(1, MAX_EPOCHS + 1):
    model.train()
    optimizer.zero_grad(set_to_none=True)
    epoch_forget_loss, epoch_retain_loss = [], []
    for step, forget_batch in enumerate(forget_loader, 1):
        try:
            retain_batch = next(retain_iterator)
        except StopIteration:
            retain_iterator = iter(retain_loader)
            retain_batch = next(retain_iterator)
        forget_batch = {key: value.to(model.device) for key, value in forget_batch.items()}
        retain_batch = {key: value.to(model.device) for key, value in retain_batch.items()}
        forget_loss = model(**forget_batch).loss
        retain_loss = model(**retain_batch).loss
        objective = (-forget_loss + RETAIN_WEIGHT * retain_loss) / GRADIENT_ACCUMULATION_STEPS
        objective.backward()
        epoch_forget_loss.append(float(forget_loss.detach()))
        epoch_retain_loss.append(float(retain_loss.detach()))
        if step % GRADIENT_ACCUMULATION_STEPS == 0 or step == len(forget_loader):
            torch.nn.utils.clip_grad_norm_(trainable, MAX_GRAD_NORM)
            optimizer.step()
            optimizer.zero_grad(set_to_none=True)

    model.config.use_cache = True
    model.eval()
    selection_forget_df = evaluate(model, forget_selection, 'forget_train', f'epoch_{epoch}', progress_every=0)
    selection_retain_df = evaluate(model, retain_selection, 'retain_train_sample', f'epoch_{epoch}', progress_every=0)
    forget_pct = percentage(selection_forget_df)
    retain_pct = percentage(selection_retain_df)
    eligible = retain_pct >= MIN_RETAIN_SELECTION
    score = (retain_pct - forget_pct) if eligible else -1000 + retain_pct - forget_pct
    row = {
        'epoch': epoch, 'mean_forget_loss': np.mean(epoch_forget_loss),
        'mean_retain_loss': np.mean(epoch_retain_loss),
        'forget_train_percentage': forget_pct,
        'retain_train_sample_percentage': retain_pct,
        'retain_eligible': eligible, 'selection_score': score,
    }
    history.append(row)
    print(row)
    if score > best_score:
        best_score, best_epoch = score, epoch
        model.save_pretrained(ADAPTER_DIR)
        tokenizer.save_pretrained(ADAPTER_DIR)
    model.config.use_cache = False
    if eligible and forget_pct <= TARGET_FORGET_MAX:
        print('Early-stop unlearning target reached.')
        break

history_df = pd.DataFrame(history)
history_df.to_csv(OUTPUT_DIR / 'unlearning_history.csv', index=False)
print('Selected epoch:', best_epoch)

## 8. Final Held-Out Evaluation

In [ ]:
del model, base_model
gc.collect()
torch.cuda.empty_cache()
model = load_base_model()
model = PeftModel.from_pretrained(model, ADAPTER_DIR)
model.eval()
after_forget_df = evaluate(model, eval_forget, 'forget', 'after_gradient_ascent')
after_retain_df = evaluate(model, eval_retain, 'retain', 'after_gradient_ascent')
after_general_df = evaluate(model, general_controls, 'general', 'after_gradient_ascent', general=True, progress_every=0)
after_forget_df.to_csv(OUTPUT_DIR / 'after_forget_results.csv', index=False)
after_retain_df.to_csv(OUTPUT_DIR / 'after_retain_results.csv', index=False)
after_general_df.to_csv(OUTPUT_DIR / 'after_general_results.csv', index=False)

all_results = pd.concat([before_forget_df, before_retain_df, before_general_df, after_forget_df, after_retain_df, after_general_df], ignore_index=True)
summary = all_results.groupby(['model_stage', 'eval_split', 'prompt_seen_in_original_training']).agg(num_questions=('contains_value', 'size'), num_correct=('contains_value', 'sum'), contains_value_percentage=('contains_value', lambda values: 100 * values.mean()), exact_match_percentage=('exact_match', lambda values: 100 * values.mean())).reset_index()
all_results.to_csv(OUTPUT_DIR / 'all_before_after_results.csv', index=False)
summary.to_csv(OUTPUT_DIR / 'percentage_summary.csv', index=False)

metrics = {
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'run_name': 'week4_gradient_ascent_unlearning_v1',
    'base_model_id': MODEL_ID,
    'source_adapter_dir': str(SOURCE_ADAPTER_DIR),
    'source_week35_metrics_path': str(STRICT_WEEK35_METRICS_PATH),
    'source_week35_run_name': strict_week35_metrics.get('run_name'),
    'source_week35_strict_scoring': strict_week35_metrics.get('strict_scoring'),
    'unlearned_adapter_dir': str(ADAPTER_DIR),
    'method': 'gradient ascent on forget loss plus gradient descent on retain loss',
    'selected_epoch': int(best_epoch),
    'training': {'max_epochs': MAX_EPOCHS, 'learning_rate': LEARNING_RATE, 'retain_weight': RETAIN_WEIGHT, 'batch_size': BATCH_SIZE, 'gradient_accumulation_steps': GRADIENT_ACCUMULATION_STEPS},
    'before_unlearning': {
        'forget_all': percentage(before_forget_df),
        'forget_heldout_paraphrases': prompt_subset_percentage(before_forget_df, False),
        'retain_all': percentage(before_retain_df),
        'retain_heldout_paraphrases': prompt_subset_percentage(before_retain_df, False),
        'general': percentage(before_general_df),
    },
    'after_unlearning': {
        'forget_all': percentage(after_forget_df),
        'forget_heldout_paraphrases': prompt_subset_percentage(after_forget_df, False),
        'retain_all': percentage(after_retain_df),
        'retain_heldout_paraphrases': prompt_subset_percentage(after_retain_df, False),
        'general': percentage(after_general_df),
    },
}
(OUTPUT_DIR / 'metrics.json').write_text(json.dumps(metrics, indent=2), encoding='utf-8')
(ADAPTER_DIR / 'unlearning_config.json').write_text(json.dumps(metrics, indent=2), encoding='utf-8')
display(summary)
print(json.dumps(metrics, indent=2))
print('Saved to:', RUN_DIR)

## Interpretation

Successful unlearning means forget accuracy decreases substantially while retain and general-control accuracy remain as high as possible. The 1,500 synthetic evaluation rows contain 500 training-identical prompts and 1,000 held-out paraphrases; the output CSV labels them separately. None of these rows or the 50 general controls is used to update the model or select a checkpoint.

In [ ]:
# GitHub output sync
commit_and_push(
    [RUN_DIR],
    'Colab: save Week 4 gradient-ascent unlearning outputs',
    repo_dir=THESIS_DIR,
)
